# Практическая работа 4. Advanced RAG

В ноутбуке исследуется влияние reranking и параметров гибридного поиска на качество RAG-системы.

В рамках работы:
- сравниваются две Cross-Encoder модели для reranking;
- анализируется влияние reranker на порядок документов и итоговые ответы;
- настраиваются веса semantic search и BM25;
- сравниваются разные embedding-модели и значения `k`;
- измеряются скорость retrieval, reranking и генерации;
- выбирается конфигурация с оптимальным балансом качества и скорости.

## Задание 1

В первом задании сравниваются две Cross-Encoder модели на одинаковых запросах: анализируется порядок документов после reranking и качество итоговых ответов.

In [1]:
!pip install -q \
    "langchain>=0.3.0" \
    "langchain-openai>=0.2.0" \
    "langchain-community>=0.3.0" \
    "langchain-core>=0.3.0" \
    "langchain-text-splitters>=0.3.0" \
    "pypdf>=5.0.0" \
    "sentence-transformers>=3.0.0" \
    "rank-bm25"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import asyncio
import time
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List

from google.colab import userdata

from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

from langchain_core.embeddings import Embeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from sentence_transformers import SentenceTransformer, CrossEncoder

/tmp/ipykernel_995/3512822923.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

OPENROUTER_MODEL = "cohere/north-mini-code:free"

_llm = ChatOpenAI(
    api_key=OPENROUTER_API_KEY,
    model=OPENROUTER_MODEL,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

Загрузка PDF и embeddings:

In [4]:
class HFEmbeddings(Embeddings):
    def __init__(
        self,
        model_name="intfloat/multilingual-e5-base",
        device="cpu",
    ):
        self.model = SentenceTransformer(
            model_name,
            device=device,
        )

    def embed_documents(self, texts: List[str]):
        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return embeddings.tolist()

    def embed_query(self, text: str):
        embedding = self.model.encode(
            [text],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]
        return embedding.tolist()

In [5]:
def load_pdf_documents(data_dir):
    pages = []

    for pdf_file in Path(data_dir).glob("*.pdf"):
        loader = PyPDFLoader(str(pdf_file))
        pages.extend(loader.load())

    return pages


def split_documents(pages):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
    )

    return splitter.split_documents(pages)

Загрузка и индексация документов:

In [6]:
DATA_DIR = "/content/docs"

pages = load_pdf_documents(DATA_DIR)
chunks = split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))

Pages: 61
Chunks: 254


Эмбеддинги и vector store:

In [7]:
embeddings = HFEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    device="cpu",
)

vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(chunks)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

['3b54929e-9c4a-4b08-b40f-88b7a6f60633',
 '4c4a274d-de68-40fe-a19c-e597bf3acf19',
 'd1bef611-f034-41a1-b776-f4987bda86fe',
 '4c012916-0a1c-43c8-be23-39fe0a6cce63',
 'a2d8d9df-8c94-4f8d-97b2-5f3e5cb00259',
 '91cc115e-817c-4d5c-99fb-ef280b8f5f2a',
 '09c569e6-8e8f-4546-8445-763c0cfa651e',
 'd46813fe-d6d6-4cc8-a2e0-e6dc8348cc0f',
 'e50f3a56-9b6d-4149-b3be-65d746f9d614',
 '569f7d55-1544-4409-8223-57ba73ad2651',
 '7af58dd3-f8ce-426f-b8fa-28df1632a236',
 '8a6d177d-b21c-4fc8-90a0-4ce7f5906b4c',
 'c0d757fd-b442-47db-8c2a-60cacccc48b8',
 '5e037e75-3846-44c7-9c62-ef6b3ba1943f',
 '483dfe1a-1061-46c1-bdba-340de5f545f9',
 '8e21883f-61ff-4661-9a94-64a027e3a342',
 'a32f7502-04dc-4b65-b23f-3690902282b9',
 'ab6ca3c5-d137-48c0-a5e3-29c1d2a51ed2',
 '30953ae3-b477-4ad1-aad9-6cf796d07713',
 '9f468d8f-23b7-4e07-99c0-a9eab194d8fd',
 'eb6d353b-0b61-4948-b951-8ce36edaaa7c',
 'ffa23ded-5014-45b3-82f2-a5a30d1e4eef',
 '4502ed3b-e506-46b2-8f56-6726c94aecc8',
 '6bce932c-f285-466c-b8f3-55999e20247a',
 '173da5c5-de20-

In [9]:
# семантический ретривер
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# bm25
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

# hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        semantic_retriever,
        bm25_retriever,
    ],
    weights=[0.5, 0.5],
)

In [10]:
# проверим, что ретривер работает

test_query = "What are the main characteristics of chronic poverty?"

retrieved_docs = hybrid_retriever.invoke(test_query)

print("Retrieved:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs[:5], start=1):
    print("\n" + "=" * 80)
    print("DOC", i)
    print(doc.page_content[:500])

Retrieved: 16

DOC 1
by measures of deficits in current consumption and expenditure. It will be important for
CPRC researchers to engage with these issues of vulnerability, both in terms of
differential vulnerabilities to shocks leading to spells of poverty, as well the manner in
which vulnerability interacts with risk aversion to hinder escape from poverty, through,
for example, adverse incorporation (Wood 2000).6
3 What are the characteristics of chronic poverty?
The defining feature of ‘chronic poverty’ is its ex

DOC 2
3
Table of contents
Title page and acknowledgements 1
Abstract 2
Table of contents 3
List of figures and list of tables 4
Abbreviations 4
1 Background 5
2 Frameworks for understanding and measuring poverty 6
2.1 Material poverty, money-metric measurement approaches and multi-
dimensional concepts of poverty
6
2.2 Absolute and relative poverty 8
2.3 Subjective poverty assessments 8
2.4 Capabilities and freedoms 9
2.5 Vulnerability 9
3 What are the characteristics of c

Загружаем два Cross-Encoder:

In [8]:
BASELINE_RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
ALTERNATIVE_RERANKER = "cross-encoder/ms-marco-MiniLM-L6-v2"

In [11]:
reranker_baseline = CrossEncoder(BASELINE_RERANKER, device="cpu")
reranker_alternative = CrossEncoder(ALTERNATIVE_RERANKER, device="cpu")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Нам важно видеть именно порядок документов после reranking, поэтому сделаем простую функцию, которая возвращает и документ, и скор:

In [12]:
def rerank_documents(query, documents, reranker, top_k=5):
    pairs = [(query, doc.page_content) for doc in documents]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

    return ranked[:top_k]

Проверка на одном запросе:

In [13]:
baseline_ranked = rerank_documents(
    test_query,
    retrieved_docs,
    reranker_baseline,
    top_k=5,
)

alternative_ranked = rerank_documents(
    test_query,
    retrieved_docs,
    reranker_alternative,
    top_k=5,
)

In [14]:
print("BASELINE RERANKER")

for i, (doc, score) in enumerate(baseline_ranked, start=1):
    print("\n" + "=" * 80)
    print(f"Rank: {i}")
    print(f"Score: {score:.4f}")
    print(doc.page_content[:400])


print("\n\nALTERNATIVE RERANKER")

for i, (doc, score) in enumerate(alternative_ranked, start=1):
    print("\n" + "=" * 80)
    print(f"Rank: {i}")
    print(f"Score: {score:.4f}")
    print(doc.page_content[:400])

BASELINE RERANKER

Rank: 1
Score: 7.4557
by measures of deficits in current consumption and expenditure. It will be important for
CPRC researchers to engage with these issues of vulnerability, both in terms of
differential vulnerabilities to shocks leading to spells of poverty, as well the manner in
which vulnerability interacts with risk aversion to hinder escape from poverty, through,
for example, adverse incorporation (Wood 2000).6
3 

Rank: 2
Score: 6.9821
means by which study of chronic poverty is presently undertaken, and will continue to
have an important role to play in research. Our understanding of chronic poverty is also
likely to draw upon notions of both absolute and relative poverty, vulnerability, social
exclusion, and capabilities and freedoms, as well as upon subjective assessments by
the poor themselves.
In the third section, the chara

Rank: 3
Score: 6.3463
human, social-cultural, social-political, financial/material and environmental/natural. An
analysis of the ways

### Сравнение Cross-Encoder моделей

Для двух reranker-моделей используется одинаковый исходный hybrid retrieval.  
Таким образом, различия в порядке документов и итоговых ответах связаны именно с этапом reranking.

Для сравнения используются три вопроса по разным PDF-документам:

- chronic poverty paper;
- information security audit report;
- PowerPoint 2010 training guide.

In [15]:
test_questions = [
    "What are the main characteristics of chronic poverty?",
    "What were the main findings and recommendations of the information security audit?",
    "What are the main features and tasks explained in the PowerPoint 2010 training guide?",
]

In [16]:
def get_doc_info(doc):
    source = Path(doc.metadata.get("source", "")).name
    page = doc.metadata.get("page", None)

    return source, page

Получаем результаты реранкера:

In [17]:
rerank_results = {}

for question in test_questions:
    print("\n" + "#" * 100)
    print("QUESTION:", question)

    # Один и тот же retrieval для обоих reranker
    retrieved_docs = hybrid_retriever.invoke(question)

    baseline_ranked = rerank_documents(
        question,
        retrieved_docs,
        reranker_baseline,
        top_k=5,
    )

    alternative_ranked = rerank_documents(
        question,
        retrieved_docs,
        reranker_alternative,
        top_k=5,
    )

    rerank_results[question] = {
        "baseline": baseline_ranked,
        "alternative": alternative_ranked,
    }

    print("\nBASELINE RERANKER")

    for rank, (doc, score) in enumerate(baseline_ranked, start=1):
        source, page = get_doc_info(doc)

        print(
            f"{rank}. score={score:.4f} | "
            f"source={source} | page={page}"
        )
        print(doc.page_content[:250].replace("\n", " "))
        print()

    print("\nALTERNATIVE RERANKER")

    for rank, (doc, score) in enumerate(alternative_ranked, start=1):
        source, page = get_doc_info(doc)

        print(
            f"{rank}. score={score:.4f} | "
            f"source={source} | page={page}"
        )
        print(doc.page_content[:250].replace("\n", " "))
        print()


####################################################################################################
QUESTION: What are the main characteristics of chronic poverty?

BASELINE RERANKER
1. score=7.4557 | source=0008-pdf.pdf | page=9
by measures of deficits in current consumption and expenditure. It will be important for CPRC researchers to engage with these issues of vulnerability, both in terms of differential vulnerabilities to shocks leading to spells of poverty, as well the 

2. score=6.9821 | source=0008-pdf.pdf | page=1
means by which study of chronic poverty is presently undertaken, and will continue to have an important role to play in research. Our understanding of chronic poverty is also likely to draw upon notions of both absolute and relative poverty, vulnerab

3. score=6.3463 | source=0008-pdf.pdf | page=1
human, social-cultural, social-political, financial/material and environmental/natural. An analysis of the ways in which severity and multi-dimensionality are often chara

Теперь используем top-5 каждого reranker как контекст для одной и той же LLM.

In [18]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the question using only the provided context.
If the context does not contain enough information, say so.
Give a concise and factual answer."""
    ),
    (
        "human",
        """Context:

{context}

Question:
{question}"""
    ),
])

In [19]:
def generate_rag_answer(question, ranked_documents):
    context = "\n\n".join(doc.page_content for doc, score in ranked_documents)

    chain = rag_prompt | _llm | StrOutputParser()

    return chain.invoke({"context": context, "question": question})

In [20]:
answer_results = []

for question in test_questions:
    print("\n" + "#" * 100)
    print("QUESTION:", question)

    baseline_answer = generate_rag_answer(
        question,
        rerank_results[question]["baseline"],
    )

    alternative_answer = generate_rag_answer(
        question,
        rerank_results[question]["alternative"],
    )

    answer_results.append({
        "question": question,
        "baseline_answer": baseline_answer,
        "alternative_answer": alternative_answer,
    })

    print("\nBASELINE ANSWER:\n")
    print(baseline_answer)

    print("\nALTERNATIVE ANSWER:\n")
    print(alternative_answer)


####################################################################################################
QUESTION: What are the main characteristics of chronic poverty?

BASELINE ANSWER:

The main characteristics of chronic poverty are:

* **Extended duration** – long‑term, persistent poverty is the defining (necessary and sufficient) feature; short‑term severe or multi‑dimensional poverty is not chronic.  
* **Severity** – chronic poverty is typically severe in its impact.  
* **Multi‑dimensionality** – it spans several domains (human, social‑cultural, social‑political, financial/material, environmental/natural) rather than being limited to income alone.  
* **Heterogeneity** – the chronically poor form a diverse group with overlapping vulnerabilities (e.g., life‑cycle stage, discrimination, health constraints, asset deprivation).  
* **Asset deprivation** – limited or low‑quality physical and financial assets, few accumulation opportunities, and high risk of asset loss, which reinforces

### Сравнение результатов reranking

| Вопрос | Совпадение top-5 | Основное изменение | Влияние на ответ |
|---|---:|---|---|
| Chronic poverty | 4/5 | Изменился порядок документов, один chunk был заменён | Небольшое улучшение |
| Information security audit | 3/5 | Alternative поднял документы с выводами и follow-up действиями | Существенное улучшение |
| PowerPoint 2010 guide | 4/5 | Alternative добавил общий раздел про presentation methods & hardware | Улучшение полноты ответа |

### Вывод

Выбор Cross-Encoder заметно повлиял на порядок документов после reranking.

Для вопроса о хронической бедности обе модели нашли почти одинаковые релевантные документы, однако альтернативный reranker выше поставил обзорные фрагменты с основными характеристиками хронической бедности. В результате итоговый ответ точнее выделил межпоколенческую передачу бедности как одну из основных характеристик.

Наиболее заметная разница появилась в вопросе об аудите информационной безопасности. Baseline reranker в основном выбрал общие фрагменты отчёта, поэтому LLM не смогла сформировать полноценный список рекомендаций. Alternative reranker поднял выше фрагменты с результатами аудита и последующими действиями, что позволило получить значительно более содержательный ответ.

Для руководства PowerPoint оба reranker нашли релевантные разделы, но alternative добавил более обзорный фрагмент о способах проведения презентаций и оборудовании. Благодаря этому итоговый ответ охватил больше тем руководства.

Таким образом, Cross-Encoder может существенно влиять на качество RAG даже при одинаковом исходном retrieval. В данном эксперименте `cross-encoder/ms-marco-MiniLM-L6-v2` в среднем формировал более полезный top-k и дал более полные итоговые ответы.

## Задание 2. Настройка гиперпараметров гибридного поиска

Сравниваются несколько конфигураций hybrid retrieval с изменением:

- весов semantic search и BM25;
- embedding-модели;
- числа возвращаемых документов `k`.

Для всех вариантов используются одинаковые вопросы, одна LLM и один Cross-Encoder reranker.  
Сравниваются выбранные документы, итоговые ответы, время поиска и время генерации.

Загрузим второй энкодер для сравнения и создадим для него свой vector store:

In [21]:
embeddings_small = HFEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    device="cpu",
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [22]:
vectorstore_small = InMemoryVectorStore(embeddings_small)
vectorstore_small.add_documents(chunks)

['ce221a7f-c87c-4473-9384-6bf923f0089a',
 'b004a981-8485-4854-8ea2-5fa1adaabce5',
 '4c3b59bb-5cb9-4b41-987d-eddf9c26058f',
 '19ea3a41-2e6f-4f6f-abb8-650f80a790dd',
 '174c5ef6-d310-43f5-8643-e78a52ad4023',
 'aea1ea06-fa39-46a7-9f62-b0132f82acf2',
 '908cb07d-5d16-424f-b122-504abed3606b',
 'e2ae7c87-1e33-44cf-b2e4-0e62bbcc9db4',
 '7f9ed90a-5bd6-49ab-b370-7faa15442a7b',
 '930abe86-0647-41df-8523-dea43b20e414',
 'bb8b8cfd-2627-43b2-bb26-7f7eb657733c',
 '51e6f53c-417f-4334-a65e-61e808ecd4f9',
 'e874429a-5501-4956-90cc-222c9507f19f',
 '34610492-6613-4f7a-835f-ce4f349e18f1',
 'a9e687f0-e00c-4a42-b172-d14e99196a13',
 'cda132ed-2363-4a6b-bada-5843c9d023f3',
 'c640f846-18c9-4751-8e99-4becdbb08026',
 '0ae7da07-8933-49c2-8e60-e8a78f217700',
 '7fe39c63-9b02-4a34-83e4-fdb5d2adcd17',
 'bb073186-be61-4ef5-b224-dc25ad3878db',
 '4254adea-b10d-437b-87e9-3df6aff1ff45',
 '073c7955-9552-4c01-b26b-de2e4a94157f',
 'ca10ce9d-94a4-4400-9594-137efc62614f',
 '620f41dd-d47c-4b70-83f2-5b98da97a823',
 '8f577bed-9dcc-

Зададим конфигурации, по которым будем проводить сравнение:

In [23]:
configs = {
    "baseline": {
        "vectorstore": vectorstore,
        "weights": [0.5, 0.5],
        "semantic_k": 10,
        "bm25_k": 10,
    },
    "semantic_heavy": {
        "vectorstore": vectorstore,
        "weights": [0.7, 0.3],
        "semantic_k": 10,
        "bm25_k": 10,
    },
    "bm25_heavy": {
        "vectorstore": vectorstore,
        "weights": [0.3, 0.7],
        "semantic_k": 10,
        "bm25_k": 10,
    },
    "small_k": {
        "vectorstore": vectorstore,
        "weights": [0.5, 0.5],
        "semantic_k": 5,
        "bm25_k": 5,
    },
    "small_encoder": {
        "vectorstore": vectorstore_small,
        "weights": [0.5, 0.5],
        "semantic_k": 10,
        "bm25_k": 10,
    },
}

Важно создавать отдельный BM25 retriever для каждой конфигурации, чтобы изменение k в одном эксперименте не влияло на другие:

In [24]:
def build_hybrid_retriever(config):
    semantic = config["vectorstore"].as_retriever(search_kwargs={"k": config["semantic_k"]})

    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = config["bm25_k"]

    hybrid = EnsembleRetriever(retrievers=[semantic, bm25], weights=config["weights"])

    return hybrid

Проверим извлечение (пока без вызова LLM). Убедимся, что все конфигурации работают и действительно выбирают разные документы.

In [25]:
retrieval_results = {}

for config_name, config in configs.items():
    print("\n" + "#" * 100)
    print("CONFIG:", config_name)

    retriever = build_hybrid_retriever(config)

    retrieval_results[config_name] = {}

    for question in test_questions:
        start = time.perf_counter()

        retrieved_docs = retriever.invoke(question)

        ranked_docs = rerank_documents(
            question,
            retrieved_docs,
            reranker_alternative,
            top_k=5,
        )

        elapsed = time.perf_counter() - start

        retrieval_results[config_name][question] = {
            "documents": ranked_docs,
            "retrieval_time_sec": elapsed,
        }

        print("\nQUESTION:", question)
        print(f"Search + reranking time: {elapsed:.3f} sec")

        for rank, (doc, score) in enumerate(ranked_docs, start=1):
            source, page = get_doc_info(doc)

            print(
                f"{rank}. source={source} | "
                f"page={page} | score={score:.4f}"
            )


####################################################################################################
CONFIG: baseline

QUESTION: What are the main characteristics of chronic poverty?
Search + reranking time: 2.550 sec
1. source=0008-pdf.pdf | page=1 | score=7.1377
2. source=0008-pdf.pdf | page=9 | score=6.7741
3. source=0008-pdf.pdf | page=1 | score=6.6921
4. source=0008-pdf.pdf | page=6 | score=5.9999
5. source=0008-pdf.pdf | page=2 | score=5.1985

QUESTION: What were the main findings and recommendations of the information security audit?
Search + reranking time: 1.335 sec
1. source=0007-pdf.pdf | page=1 | score=3.3302
2. source=0007-pdf.pdf | page=1 | score=1.5758
3. source=0007-pdf.pdf | page=3 | score=0.8999
4. source=0007-pdf.pdf | page=6 | score=-0.8618
5. source=0007-pdf.pdf | page=4 | score=-1.5029

QUESTION: What are the main features and tasks explained in the PowerPoint 2010 training guide?
Search + reranking time: 1.690 sec
1. source=0009-pdf.pdf | page=12 | score=2.0592


Главный вывод на этом этапе: веса 0.5/0.5, 0.7/0.3, 0.3/0.7 и смена multilingual-e5-base на multilingual-e5-small почти не изменили итоговый top-5 после reranking. А вот уменьшение k до 5 заметно изменило набор кандидатов и одновременно почти вдвое ускорило поиск.

### Генерация ответов для разных конфигураций

Для каждого варианта hybrid retrieval используется одинаковая LLM и одинаковый Cross-Encoder reranker.

Сравниваются:
- итоговый top-5 документов;
- время retrieval + reranking;
- время генерации;
- содержание финального ответа.

Вызовем LLM для всех сохранённых результатов:

In [26]:
generation_results = []

for config_name, questions_data in retrieval_results.items():
    print("\n" + "#" * 100)
    print("CONFIG:", config_name)

    for question in test_questions:
        ranked_docs = questions_data[question]["documents"]
        retrieval_time = questions_data[question]["retrieval_time_sec"]

        start = time.perf_counter()

        answer = generate_rag_answer(question, ranked_docs)

        generation_time = time.perf_counter() - start

        generation_results.append({
            "config": config_name,
            "question": question,
            "retrieval_time_sec": retrieval_time,
            "generation_time_sec": generation_time,
            "total_time_sec": retrieval_time + generation_time,
            "answer": answer,
        })

        print("\nQUESTION:", question)
        print(f"Retrieval + reranking: {retrieval_time:.3f} sec")
        print(f"Generation: {generation_time:.3f} sec")
        print(f"Total: {retrieval_time + generation_time:.3f} sec")

        print("\nANSWER:\n")
        print(answer)


####################################################################################################
CONFIG: baseline

QUESTION: What are the main characteristics of chronic poverty?
Retrieval + reranking: 2.550 sec
Generation: 6.359 sec
Total: 8.909 sec

ANSWER:

**Main characteristics of chronic poverty (as outlined in the context):**

- **Long duration** – Poverty must persist over an extended period; duration is both necessary and sufficient for a condition to be classified as chronic.  
- **Multi‑dimensionality** – Chronic poverty is typically not just income‑based but spans several dimensions (e.g., human, social‑cultural, social‑political, financial/material, environmental/natural).  
- **Severity** – It is usually severe (often termed “ultra‑poverty”) and the deficits in consumption and expenditure are substantial.  
- **Intergenerational transmission** – Poverty can be passed across generations, reinforcing its persistence.  
- **Vulnerability & overlapping risks** – Those in

In [27]:
generation_df = pd.DataFrame(generation_results)

summary_df = generation_df[
    [
        "config",
        "question",
        "retrieval_time_sec",
        "generation_time_sec",
        "total_time_sec",
    ]
]

summary_df

,config,question,retrieval_time_sec,generation_time_sec,total_time_sec
0,baseline,What are the main characteristics of chronic p...,2.550131,6.358692,8.908822
1,baseline,What were the main findings and recommendation...,1.334650,7.101598,8.436248
2,baseline,What are the main features and tasks explained...,1.690213,8.072563,9.762776
3,semantic_heavy,What are the main characteristics of chronic p...,1.572504,5.614628,7.187132
4,semantic_heavy,What were the main findings and recommendation...,1.334530,5.712463,7.046993
5,semantic_heavy,What are the main features and tasks explained...,1.700773,7.066668,8.767441
6,bm25_heavy,What are the main characteristics of chronic p...,1.560460,8.869087,10.429548
7,bm25_heavy,What were the main findings and recommendation...,1.832924,9.605585,11.438509
8,bm25_heavy,What are the main features and tasks explained...,2.526252,4.580165,7.106417
9,small_k,What are the main characteristics of chronic p...,0.863000,5.703732,6.566731


In [28]:
# таблица со средними

avg_summary_df = (
    generation_df
    .groupby("config")[
        [
            "retrieval_time_sec",
            "generation_time_sec",
            "total_time_sec",
        ]
    ]
    .mean()
    .reset_index()
)

avg_summary_df

,config,retrieval_time_sec,generation_time_sec,total_time_sec
0,baseline,1.858331,7.177618,9.035948
1,bm25_heavy,1.973212,7.684946,9.658158
2,semantic_heavy,1.535935,6.131253,7.667189
3,small_encoder,1.487307,4.746520,6.233828
4,small_k,0.825217,5.208221,6.033438


### Сравнение конфигураций

| Конфигурация | Среднее retrieval + reranking | Среднее generation | Среднее total | Изменение качества |
|---|---:|---:|---:|---|
| baseline | 1.86 с | 7.18 с | 9.04 с | Базовый уровень |
| semantic_heavy | 1.54 с | 6.13 с | 7.67 с | Существенных изменений нет |
| bm25_heavy | 1.97 с | 7.68 с | 9.66 с | Существенных изменений нет |
| small_encoder | 1.49 с | 4.75 с | 6.23 с | Качество практически сохранилось |
| small_k | 0.83 с | 5.21 с | 6.03 с | Качество в целом сохранилось |

### Вывод

В эксперименте изменение весов semantic search и BM25 с `0.5/0.5` на `0.7/0.3` и `0.3/0.7` почти не повлияло на итоговый набор документов после reranking. Это показывает, что Cross-Encoder на данном наборе данных сглаживает различия между результатами двух retriever-ов.

Замена `multilingual-e5-base` на более лёгкий `multilingual-e5-small` также практически не изменила top-5 документов, но немного уменьшила среднее время выполнения.

Наиболее заметный эффект дало уменьшение `k` с 10 до 5. Retrieval + reranking ускорился примерно в два раза, а итоговые ответы остались содержательно сопоставимыми с baseline. В отдельных случаях контекст немного изменился, но существенной потери качества не произошло.

Таким образом, для данного набора документов оптимальной конфигурацией оказался вариант с `k=5` для semantic retriever и BM25 при весах `0.5/0.5` и использовании `multilingual-e5-base`. Он обеспечивает лучший баланс между скоростью и качеством ответа.

Более лёгкий `multilingual-e5-small` также показал хорошие результаты и может быть предпочтителен при более строгих ограничениях по вычислительным ресурсам.